In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse


In [3]:
# Monarch/Monarch_final/Mouse/Gene_Mouse_Phenotype.csv
# hmdhp/hmdhp_MOUSE_GENE_PHENOTYPE.csv
# mgi_do/mgido_MOUSE_GENE_PHENOTYPE.csv

In [4]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Mouse_gene_phenotype
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse/Mouse_gene_phenotype/Mouse_gene_phenotype.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

mkdir: cannot create directory ‘Mouse_gene_phenotype’: File exists


In [5]:
# MP_phenotype_mapp = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/data_collection/monarchkg/monarch-kg_nodes.tsv',sep = '\t')
# MP_phenotype_mapp



# MP_phenotype_mapp = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/data_collection/mgi_do/MPheno_OBO.ontology')
# MP_phenotype_mapp

import pandas as pd
from collections import defaultdict

obo_file = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/mgi_do/MPheno_OBO.ontology'

terms = []
current = defaultdict(list)

with open(obo_file, "r", encoding="utf-8") as f:

    for line in f:
        line = line.strip()

        # new term starts
        if line == "[Term]":

            # save previous term
            if current:
                terms.append(dict(current))

            current = defaultdict(list)

        elif ": " in line:

            key, value = line.split(": ", 1)

            current[key].append(value)

# append last term
if current:
    terms.append(dict(current))

# convert to dataframe
rows = []

for t in terms:

    rows.append({
        "id": t.get("id", [None])[0],
        "name": t.get("name", [None])[0],
        "definition": t.get("def", [None])[0],
        "synonyms": " | ".join(t.get("synonym", [])),
        "is_a": " | ".join(t.get("is_a", [])),
        "alt_ids": " | ".join(t.get("alt_id", [])),
        "obsolete": t.get("is_obsolete", [False])[0]
    })

df = pd.DataFrame(rows)

df
MP_dict = dict(zip(df['name'], df['id']))
MP_dict

{None: None,
 'mammalian phenotype': 'MP:0000001',
 'obsolete Morphology': 'MP:0000002',
 'abnormal adipose tissue morphology': 'MP:0000003',
 'increased brown adipose tissue amount': 'MP:0000005',
 'increased white adipose tissue amount': 'MP:0000008',
 'abnormal abdominal fat pad morphology': 'MP:0000010',
 'obsolete loss of subcutaneous adipose tissue': 'MP:0000012',
 'abnormal adipose tissue distribution': 'MP:0000013',
 'abnormal ear pigmentation': 'MP:0000015',
 'big ears': 'MP:0000017',
 'small ears': 'MP:0000018',
 'thick ears': 'MP:0000019',
 'scaly ears': 'MP:0000020',
 'prominent ears': 'MP:0000021',
 'abnormal ear shape': 'MP:0000022',
 'abnormal ear position': 'MP:0000023',
 'lowered ear position': 'MP:0000024',
 'otic hypertelorism': 'MP:0000025',
 'abnormal inner ear morphology': 'MP:0000026',
 'obsolete horizontal canal defects': 'MP:0000027',
 'abnormal pars superior vestibularis morphology': 'MP:0000028',
 'abnormal malleus morphology': 'MP:0000029',
 'abnormal tympan

In [6]:
df[df['name'].str.contains('skeleton phenotype', case=False, na=False)]
MP_dict.get('MP:0005390')

In [7]:
# Mouse = pd.read_csv(
#     f'{BASE_DIR}data_collection/databases_for_mapping/ncbi/Mus_musculus.gene_info',
#     sep='\t',
    
# )
# # Mouse["LocusTag"] = Mouse["LocusTag"].str.replace("Dmel_", "", regex=False)

# # Mouse_symbol2Locus_dict = dict(zip(Mouse['Symbol'], Mouse['LocusTag']))
# # Mouse
# # Mouse_symbol2Locus_dict
# Mouse

# MGIDO # because HMDP is taken from MGI_DO source only

In [8]:
hmdp = pd.read_csv(PROC_DIR + 'hmdhp/hmdhp_MOUSE_GENE_PHENOTYPE.csv')
hmdp.columns = hmdp.columns.str.lower()
hmdp = hmdp[~hmdp['head_detail_name'].isna()]
hmdp['tail'] = (
    hmdp['tail_detail_name']
    .str.lower()
    .str.strip()
    .map(MP_dict)
)
hmdp['species'] = 'M.musculus'
print(f"hmdp: {len(hmdp):,} rows")
hmdp['head_id_is'] = 'NCBI_ID'
hmdp['tail_id_is'] = 'MP_ID'

hmdp = hmdp[~hmdp['tail'].isna()]
hmdp

hmdp: 76,341 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,1110059G10Rik,Gene_Phenotype,MP:0005390,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1110059G10 gene,skeleton phenotype,M.musculus
1,1500009L16Rik,Gene_Phenotype,MP:0005376,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,homeostasis/metabolism phenotype,M.musculus
2,1500009L16Rik,Gene_Phenotype,MP:0005390,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,skeleton phenotype,M.musculus
3,1700001O22Rik,Gene_Phenotype,MP:0005385,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1700001O22 gene,cardiovascular system phenotype,M.musculus
4,1700001O22Rik,Gene_Phenotype,MP:0003631,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1700001O22 gene,nervous system phenotype,M.musculus
...,...,...,...,...,...,...,...,...,...,...,...,...
76336,mt-Rnr2,Gene_Phenotype,MP:0005385,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,16S ribosomal RNA,cardiovascular system phenotype,M.musculus
76337,mt-Rnr2,Gene_Phenotype,MP:0005378,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,16S ribosomal RNA,growth/size/body region phenotype,M.musculus
76338,mt-Rnr2,Gene_Phenotype,MP:0010771,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,16S ribosomal RNA,integument phenotype,M.musculus
76339,mt-Rnr2,Gene_Phenotype,MP:0010768,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,16S ribosomal RNA,mortality/aging,M.musculus


# Mgi_DO

In [9]:
mgido = pd.read_csv(PROC_DIR + 'mgi_do/mgido_MOUSE_GENE_PHENOTYPE.csv')
mgido.columns = mgido.columns.str.lower()
mgido = mgido[~mgido['head_detail_name'].isna()]
cols = ['tail', 'tail_detail_name']

mgido['tail'] = (
    hmdp['tail_detail_name']
    .str.lower()
    .str.strip()
    .map(MP_dict)
)

mgido['kg_type'] = 'Generalised'
mgido['kg_source'] = 'MGI_DO'

mgido['species'] = 'M.musculus'
print(f"mgido: {len(mgido):,} rows")
mgido['head_id_is'] = 'NCBI_ID'
mgido['tail_id_is'] = 'MP_ID'

mgido = mgido[~mgido['tail'].isna()]
mgido

mgido: 282,888 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,1110059G10Rik,Gene_Phenotype,MP:0005390,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1110059G10 gene,vertebral transformation,Generalised,M.musculus
1,1500009L16Rik,Gene_Phenotype,MP:0005376,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,decreased bone mineral density,Generalised,M.musculus
2,1500009L16Rik,Gene_Phenotype,MP:0005390,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,increased circulating alkaline phosphatase level,Generalised,M.musculus
3,1500009L16Rik,Gene_Phenotype,MP:0005385,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,increased circulating aspartate transaminase l...,Generalised,M.musculus
4,1500009L16Rik,Gene_Phenotype,MP:0003631,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,increased circulating calcium level,Generalised,M.musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...
76336,Epb41,Gene_Phenotype,MP:0005385,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,erythrocyte membrane protein band 4.1,enlarged spleen,Generalised,M.musculus
76337,Epb41,Gene_Phenotype,MP:0005378,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,erythrocyte membrane protein band 4.1,extramedullary hematopoiesis,Generalised,M.musculus
76338,Epb41,Gene_Phenotype,MP:0010771,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,erythrocyte membrane protein band 4.1,hemolytic anemia,Generalised,M.musculus
76339,Epb41,Gene_Phenotype,MP:0010768,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,erythrocyte membrane protein band 4.1,increased circulating aspartate transaminase l...,Generalised,M.musculus


In [12]:
mgido[mgido['tail'].str.startswith('MP:')]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,1110059G10Rik,Gene_Phenotype,MP:0005390,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1110059G10 gene,vertebral transformation,Generalised,M.musculus
1,1500009L16Rik,Gene_Phenotype,MP:0005376,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,decreased bone mineral density,Generalised,M.musculus
2,1500009L16Rik,Gene_Phenotype,MP:0005390,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,increased circulating alkaline phosphatase level,Generalised,M.musculus
3,1500009L16Rik,Gene_Phenotype,MP:0005385,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,increased circulating aspartate transaminase l...,Generalised,M.musculus
4,1500009L16Rik,Gene_Phenotype,MP:0003631,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,increased circulating calcium level,Generalised,M.musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...
76336,Epb41,Gene_Phenotype,MP:0005385,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,erythrocyte membrane protein band 4.1,enlarged spleen,Generalised,M.musculus
76337,Epb41,Gene_Phenotype,MP:0005378,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,erythrocyte membrane protein band 4.1,extramedullary hematopoiesis,Generalised,M.musculus
76338,Epb41,Gene_Phenotype,MP:0010771,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,erythrocyte membrane protein band 4.1,hemolytic anemia,Generalised,M.musculus
76339,Epb41,Gene_Phenotype,MP:0010768,Gene,NaN,Phenotype,MGI_DO,NCBI_ID,MP_ID,erythrocyte membrane protein band 4.1,increased circulating aspartate transaminase l...,Generalised,M.musculus


# Monarch

In [13]:
monarch = pd.read_csv(PROC_DIR + 'Monarch/Monarch_final/Mouse/Gene_Mouse_Phenotype.csv')

monarch.columns = monarch.columns.str.lower()
monarch = monarch[~monarch['head_detail_name'].isna()]
monarch['kg_type'] = 'Generalised'
monarch['kg_source'] = 'Monarch_KG'
monarch['species'] = 'M.musculus'
monarch['head_id_is'] = 'NCBI_ID'
monarch['tail_id_is'] = 'MP_ID'
print(f"monarch: {len(monarch):,} rows")
monarch

monarch: 280,662 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_name,species,kg_type,kg_source
0,Gabarapl2,Gene_Phenotype,MP:0005016,Gene,has_phenotype,Phenotype,infores:mgi,GABA type A receptor associated protein like 2,decreased lymphocyte cell number,Mus musculus,NaN,NCBI_ID,MP_ID,Gabarapl2,M.musculus,Generalised,Monarch_KG
1,Gabarapl2,Gene_Phenotype,MP:0005419,Gene,has_phenotype,Phenotype,infores:mgi,GABA type A receptor associated protein like 2,decreased circulating serum albumin level,Mus musculus,NaN,NCBI_ID,MP_ID,Gabarapl2,M.musculus,Generalised,Monarch_KG
2,Gabarapl2,Gene_Phenotype,MP:0011100,Gene,has_phenotype,Phenotype,infores:mgi,GABA type A receptor associated protein like 2,"preweaning lethality, complete penetrance",Mus musculus,NaN,NCBI_ID,MP_ID,Gabarapl2,M.musculus,Generalised,Monarch_KG
3,Osr2,Gene_Phenotype,MP:0000030,Gene,has_phenotype,Phenotype,infores:mgi,odd-skipped related 2,abnormal tympanic ring morphology,Mus musculus,NaN,NCBI_ID,MP_ID,Osr2,M.musculus,Generalised,Monarch_KG
4,Osr2,Gene_Phenotype,MP:0000030,Gene,has_phenotype,Phenotype,infores:mgi,odd-skipped related 2,abnormal tympanic ring morphology,Mus musculus,NaN,NCBI_ID,MP_ID,Osr2,M.musculus,Generalised,Monarch_KG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286519,Msh3,Gene_Phenotype,MP:0002135,Gene,has_phenotype,Phenotype,infores:mgi,mutS homolog 3,abnormal kidney morphology,Mus musculus,NaN,NCBI_ID,MP_ID,Msh3,M.musculus,Generalised,Monarch_KG
286520,Msh3,Gene_Phenotype,MP:0002699,Gene,has_phenotype,Phenotype,infores:mgi,mutS homolog 3,abnormal vitreous body morphology,Mus musculus,NaN,NCBI_ID,MP_ID,Msh3,M.musculus,Generalised,Monarch_KG
286521,Msh3,Gene_Phenotype,MP:0003068,Gene,has_phenotype,Phenotype,infores:mgi,mutS homolog 3,enlarged kidney,Mus musculus,NaN,NCBI_ID,MP_ID,Msh3,M.musculus,Generalised,Monarch_KG
286522,Htr2a,Gene_Phenotype,MP:0000479,Gene,has_phenotype,Phenotype,infores:mgi,5-hydroxytryptamine (serotonin) receptor 2A,abnormal enterocyte morphology,Mus musculus,NaN,NCBI_ID,MP_ID,Htr2a,M.musculus,Generalised,Monarch_KG


In [14]:
monarch[monarch['tail_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_name,species,kg_type,kg_source


# Consolidate into Unified Schem

In [15]:
# List all source DataFrames to include
source_dfs = [
    monarch, mgido, hmdp
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 433,344


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,Gabarapl2,Gene_Phenotype,MP:0005016,Gene,has_phenotype,Phenotype,Monarch_KG,Generalised,NCBI_ID,MP_ID,GABA type A receptor associated protein like 2,decreased lymphocyte cell number,M.musculus
1,Gabarapl2,Gene_Phenotype,MP:0005419,Gene,has_phenotype,Phenotype,Monarch_KG,Generalised,NCBI_ID,MP_ID,GABA type A receptor associated protein like 2,decreased circulating serum albumin level,M.musculus
2,Gabarapl2,Gene_Phenotype,MP:0011100,Gene,has_phenotype,Phenotype,Monarch_KG,Generalised,NCBI_ID,MP_ID,GABA type A receptor associated protein like 2,"preweaning lethality, complete penetrance",M.musculus
3,Osr2,Gene_Phenotype,MP:0000030,Gene,has_phenotype,Phenotype,Monarch_KG,Generalised,NCBI_ID,MP_ID,odd-skipped related 2,abnormal tympanic ring morphology,M.musculus
4,Osr2,Gene_Phenotype,MP:0000030,Gene,has_phenotype,Phenotype,Monarch_KG,Generalised,NCBI_ID,MP_ID,odd-skipped related 2,abnormal tympanic ring morphology,M.musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...
433339,mt-Rnr2,Gene_Phenotype,MP:0005385,Gene,NaN,Phenotype,MGI_DO,None,NCBI_ID,MP_ID,16S ribosomal RNA,cardiovascular system phenotype,M.musculus
433340,mt-Rnr2,Gene_Phenotype,MP:0005378,Gene,NaN,Phenotype,MGI_DO,None,NCBI_ID,MP_ID,16S ribosomal RNA,growth/size/body region phenotype,M.musculus
433341,mt-Rnr2,Gene_Phenotype,MP:0010771,Gene,NaN,Phenotype,MGI_DO,None,NCBI_ID,MP_ID,16S ribosomal RNA,integument phenotype,M.musculus
433342,mt-Rnr2,Gene_Phenotype,MP:0010768,Gene,NaN,Phenotype,MGI_DO,None,NCBI_ID,MP_ID,16S ribosomal RNA,mortality/aging,M.musculus


# Sanity Check — Distinct Values

In [16]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Gene_Phenotype'}
head_type           : {'Gene'}
tail_type           : {'Phenotype'}
relation_type       : {nan, 'has_phenotype'}
kg_source           : {'Monarch_KG', 'MGI_DO'}
head_id_is          : {'NCBI_ID'}
tail_id_is          : {'MP_ID'}


In [17]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 433,344 remaining


# NaN Audit (pre-dedup)

In [18]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,152682,152682,305364
tail_type,0,0,0
kg_source,0,0,0
kg_type,76341,0,76341
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [19]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 433,344  |  After dedup: 343,694


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,1110059G10Rik,Gene_Phenotype,MP:0003036,Gene,has_phenotype,Phenotype,Monarch_KG,Generalised,NCBI_ID,MP_ID,RIKEN cDNA 1110059G10 gene,vertebral transformation,M.musculus
1,1110059G10Rik,Gene_Phenotype,MP:0005390,Gene,None,Phenotype,MGI_DO,Generalised,NCBI_ID,MP_ID,RIKEN cDNA 1110059G10 gene,vertebral transformation,M.musculus
2,1500009L16Rik,Gene_Phenotype,MP:0000063,Gene,has_phenotype,Phenotype,Monarch_KG,Generalised,NCBI_ID,MP_ID,RIKEN cDNA 1500009L16 gene,decreased bone mineral density,M.musculus


In [20]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,101021,0,101021
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [21]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'Monarch_KG', 'MGI_DO', 'MGI_DO::Monarch_KG'} kg_source
Monarch_KG            242563
MGI_DO                101021
MGI_DO::Monarch_KG       110
Name: count, dtype: int64


In [22]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'', 'Generalised'} kg_type
Generalised    281332
                62362
Name: count, dtype: int64


In [23]:
final_df_dedup['tail'].unique()

array(['MP:0003036', 'MP:0005390', 'MP:0000063', ..., 'MP:0006103',
       'MP:0000407', 'MP:0013173'], shape=(10880,), dtype=object)

In [24]:
unique_vals = final_df_dedup['tail'].dropna().unique()

with open('tail_unique.txt', 'w') as f:
    for val in unique_vals:
        f.write(f"{val}\n")

In [25]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse/Mouse_gene_phenotype/Mouse_gene_phenotype.csv'